# 查看数据集目录结构

In [2]:
import seedir as sd
sd.seedir('Watermelon87_Semantic_Seg_Labelme', style='emoji', depthlimit=1)

📁 Watermelon87_Semantic_Seg_Labelme/
├─📁 images/
└─📁 labelme_jsons/


## 删除系统自动生成的多余文件

在处理跨平台共享的数据或项目时，经常会产生一些不需要的隐藏文件和缓存文件夹。
本项目主要清理以下三类无用数据：
* `__MACOSX/`: Mac 系统压缩文件时自动生成的资源分叉文件夹。
* `.DS_Store`: Mac 系统用于存储文件夹自定义属性的隐藏文件。
* `.ipynb_checkpoints/`: Jupyter Notebook 自动保存的检查点备份文件夹。

使用跨平台的 Python 原生库 `pathlib` 和 `shutil` 来执行，确保在 Windows 环境下也能顺利运行。

### 查看待删除的多余文件

In [3]:
import os
import shutil
from pathlib import Path

# 定义待清理的无用文件/文件夹名称列表
junk_names = ['__MACOSX', '.DS_Store', '.ipynb_checkpoints']
base_path = Path('.')

print("🔍 发现以下待清理的多余文件/文件夹：")
found_junks = []

# 遍历当前目录及子目录查找
for name in junk_names:
    # rglob 用于递归搜索
    for item in base_path.rglob(name):
        found_junks.append(item)
        print(f" - {item}")

if not found_junks:
    print("   未发现多余文件，目录很干净！")

🔍 发现以下待清理的多余文件/文件夹：
 - .ipynb_checkpoints


### 执行删除操作
确认上方列出的文件无误后，运行下方代码彻底删除它们。**注意：此操作不可逆。**

In [4]:
print("🗑️ 开始清理...")
deleted_count = 0

for item in found_junks:
    if item.exists():
        try:
            if item.is_dir():
                shutil.rmtree(item) # 删除文件夹及其内部所有内容
            else:
                item.unlink()       # 删除单个文件
            print(f" ✅ 已删除: {item}")
            deleted_count += 1
        except Exception as e:
            print(f" ❌ 删除失败 {item}: {e}")

print(f"\n✨ 清理完成！共删除了 {deleted_count} 个多余文件/文件夹。")

# 再次验证
print("🔍 再次验证残留...")
remain_count = sum(1 for name in junk_names for _ in base_path.rglob(name))
if remain_count == 0:
    print("   验证通过，多余文件已全部清除干净！")
else:
    print(f"   注意：还有 {remain_count} 个文件未能删除，请检查文件是否被占用。")

🗑️ 开始清理...
 ✅ 已删除: .ipynb_checkpoints

✨ 清理完成！共删除了 1 个多余文件/文件夹。
🔍 再次验证残留...
   验证通过，多余文件已全部清除干净！


# 导入模块

In [5]:
import os
import json
import numpy as np
import cv2
import shutil
from tqdm import tqdm

# 数据集及类别信息

In [6]:
Dataset_Path = 'Watermelon87_Semantic_Seg_Labelme'

# 1.每个类别的信息及画mask的顺序（按照由大到小，由粗到精的顺序）

In [7]:
# 0-背景，从 1 开始
class_info = [
    {'label':'red', 'type':'polygon', 'color':1},                    # polygon 多段线
    {'label':'green', 'type':'polygon', 'color':2},
    {'label':'white', 'type':'polygon', 'color':3},
    {'label':'seed-black','type':'polygon','color':4},
    {'label':'seed-white','type':'polygon','color':5}
]

# 2.单张图像labelme转mask函数

In [8]:
def labelme2mask_single_img(img_path, labelme_json_path):
    '''
    输入原始图像路径和labelme标注路径，输出 mask
    '''
    
    img_bgr = cv2.imread(img_path)
    img_mask = np.zeros(img_bgr.shape[:2]) # 创建空白图像 0-背景
    
    with open(labelme_json_path, 'r', encoding='utf-8') as f:
        labelme = json.load(f)
        
    for one_class in class_info: # 按顺序遍历每一个类别
        for each in labelme['shapes']: # 遍历所有标注，找到属于当前类别的标注
            if each['label'] == one_class['label']:
                if one_class['type'] == 'polygon': # polygon 多段线标注

                    # 获取点的坐标
                    points = [np.array(each['points'], dtype=np.int32).reshape((-1, 1, 2))]

                    # 在空白图上画 mask（闭合区域）
                    img_mask = cv2.fillPoly(img_mask, points, color=one_class['color'])

                elif one_class['type'] == 'line' or one_class['type'] == 'linestrip': # line 或者 linestrip 线段标注

                    # 获取点的坐标
                    points = [np.array(each['points'], dtype=np.int32).reshape((-1, 1, 2))]

                    # 在空白图上画 mask（非闭合区域）
                    img_mask = cv2.polylines(img_mask, points, isClosed=False, color=one_class['color'], thickness=one_class['thickness']) 

                elif one_class['type'] == 'circle': # circle 圆形标注

                    points = np.array(each['points'], dtype=np.int32)

                    center_x, center_y = points[0][0], points[0][1] # 圆心点坐标

                    edge_x, edge_y = points[1][0], points[1][1]     # 圆周点坐标

                    radius = np.linalg.norm(np.array([center_x, center_y] - np.array([edge_x, edge_y]))).astype('uint32') # 半径

                    img_mask = cv2.circle(img_mask, (center_x, center_y), radius, one_class['color'], one_class['thickness'])

                else:
                    print('未知标注类型', one_class['type'])
                    
    return img_mask

# 3.批量运行（对每张图片执行labelme转mask）

In [9]:
os.chdir(Dataset_Path)
os.mkdir('masks')
os.chdir('images')

In [10]:
for img_path in tqdm(os.listdir()):
    
    try:
    
        labelme_json_path = os.path.join('../', 'labelme_jsons', '.'.join(img_path.split('.')[:-1])+'.json')

        img_mask = labelme2mask_single_img(img_path, labelme_json_path)

        mask_path = img_path.split('.')[0] + '.png'

        cv2.imwrite(os.path.join('../','masks',mask_path), img_mask)
    
    except Exception as E:
        print(img_path, '转换失败', E)

100%|█████████████████████████████████████████████████████████████████████████████████| 87/87 [00:00<00:00, 118.36it/s]


# 4.将转换后的mask保存到masks文件夹中

In [13]:
import os
import shutil

# 直接使用前面定义好的数据集根目录，避免使用 os.chdir() 跳来跳去
Dataset_Path = 'Watermelon87_Semantic_Seg_Labelme'

# 定义目标文件夹的完整路径
images_path = os.path.join(Dataset_Path, 'images')
img_dir_path = os.path.join(Dataset_Path, 'img_dir')

masks_path = os.path.join(Dataset_Path, 'masks')
ann_dir_path = os.path.join(Dataset_Path, 'ann_dir')

labelme_jsons_path = os.path.join(Dataset_Path, 'labelme_jsons')

print("🚀 开始整理目录结构...")

# 1. 重命名 images -> img_dir (加入判断，防止重复运行报错)
if os.path.exists(images_path):
    shutil.move(images_path, img_dir_path)
    print(" ✅ 'images' 已成功重命名为 'img_dir'")
elif os.path.exists(img_dir_path):
    print(" 💡 'img_dir' 已经存在，无需重复重命名")
else:
    print(" ⚠️ 找不到 'images' 或 'img_dir'，请检查前面步骤是否正常。")

# 2. 重命名 masks -> ann_dir
if os.path.exists(masks_path):
    shutil.move(masks_path, ann_dir_path)
    print(" ✅ 'masks' 已成功重命名为 'ann_dir'")
elif os.path.exists(ann_dir_path):
    print(" 💡 'ann_dir' 已经存在，无需重复重命名")
else:
    print(" ⚠️ 找不到 'masks' 或 'ann_dir'，请检查前面步骤是否正常。")

# 3. 删除 labelme_jsons 文件夹
if os.path.exists(labelme_jsons_path):
    shutil.rmtree(labelme_jsons_path)
    print(" ✅ 'labelme_jsons' 文件夹已彻底清理")
else:
    print(" 💡 'labelme_jsons' 文件夹已被清理，无需重复操作")

print("✨ 所有文件夹重命名与清理步骤完美完成！")

🚀 开始整理目录结构...
 ⚠️ 找不到 'images' 或 'img_dir'，请检查前面步骤是否正常。
 ⚠️ 找不到 'masks' 或 'ann_dir'，请检查前面步骤是否正常。
 💡 'labelme_jsons' 文件夹已被清理，无需重复操作
✨ 所有文件夹重命名与清理步骤完美完成！


# 5.得到最终的语义分割数据集

In [14]:
# 查看数据集的目录结构
import seedir as sd
sd.seedir('Watermelon87_Semantic_Seg_Labelme', style='emoji', depthlimit=1)

📄 Watermelon87_Semantic_Seg_Labelme
